# Exercise 12: Cross validation

In this exercise, we'll practice implementing cross validation techniques, including leave-one-out and k-fold cross validation. We'll use the `PimaIndiansDiabetes2` practice dataset, which has medical data on a group of Pima Native American women, including whether or not they have diabetes. This dataset is part of the `mlbench` package. We'll be using each person's medical history to predict whether or not they have been diagnosed with diabetes.

---
## 1. Load Data (1 point)

Load the `tidyverse`, `boot`, and `mlbench` packages (you may need to install `boot` and `mlbench`).

Load the `PimaIndiansDiabetes2` dataset using the `data()` function. Drop the `insulin` column (it just has a lot of missing data) and then drop `NA`s from the rest of the dataset. Save your updated dataset to a new variable name. Finally, print the dimensions of your new dataset, and look at the first few lines of data.

In [2]:
install.packages('mlbench')
need_library <- c('tidyverse', 'boot', 'mlbench')
lapply(need_library, library, character = TRUE)

data(PimaIndiansDiabetes2)
df <- PimaIndiansDiabetes2
df <- df %>% select(-insulin) %>% na.omit()

str(df)
head(df)


The downloaded binary packages are in
	/var/folders/5_/v0q2657s7b354r1303qrlrxw0000gp/T//RtmpMyZvWQ/downloaded_packages


[[1]]
 [1] "mlbench"   "boot"      "lubridate" "forcats"   "stringr"   "dplyr"    
 [7] "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"   "tidyverse"
[13] "stats"     "graphics"  "grDevices" "utils"     "datasets"  "methods"  
[19] "base"     

[[2]]
 [1] "mlbench"   "boot"      "lubridate" "forcats"   "stringr"   "dplyr"    
 [7] "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"   "tidyverse"
[13] "stats"     "graphics"  "grDevices" "utils"     "datasets"  "methods"  
[19] "base"     

[[3]]
 [1] "mlbench"   "boot"      "lubridate" "forcats"   "stringr"   "dplyr"    
 [7] "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"   "tidyverse"
[13] "stats"     "graphics"  "grDevices" "utils"     "datasets"  "methods"  
[19] "base"

'data.frame':	532 obs. of  8 variables:
 $ pregnant: num  6 1 1 0 3 2 1 5 0 1 ...
 $ glucose : num  148 85 89 137 78 197 189 166 118 103 ...
 $ pressure: num  72 66 66 40 50 70 60 72 84 30 ...
 $ triceps : num  35 29 23 35 32 45 23 19 47 38 ...
 $ mass    : num  33.6 26.6 28.1 43.1 31 30.5 30.1 25.8 45.8 43.3 ...
 $ pedigree: num  0.627 0.351 0.167 2.288 0.248 ...
 $ age     : num  50 31 21 33 26 53 59 51 31 33 ...
 $ diabetes: Factor w/ 2 levels "neg","pos": 2 1 1 2 2 2 2 2 2 1 ...
 - attr(*, "na.action")= 'omit' Named int [1:236] 3 6 8 10 11 12 13 16 18 22 ...
  ..- attr(*, "names")= chr [1:236] "3" "6" "8" "10" ...


,pregnant,glucose,pressure,triceps,mass,pedigree,age,diabetes
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
1,6,148,72,35,33.6,0.627,50,pos
2,1,85,66,29,26.6,0.351,31,neg
4,1,89,66,23,28.1,0.167,21,neg
5,0,137,40,35,43.1,2.288,33,pos
7,3,78,50,32,31.0,0.248,26,pos
9,2,197,70,45,30.5,0.158,53,pos


(Note that in medical contexts, `pedigree` refers to a system of measuring family history of a condition. So here, higher numbers mean greater family history of diabetes. You can read more about this dataset [here](https://rdrr.io/cran/mlbench/man/PimaIndiansDiabetes.html).)

---
## 2. Leave-one-out Cross Validation (4 points)

In the tutorial, we learned how to fit leave-one-out cross validation using the `cv.glm` function from the `boot` package. But we can also do this manually using `predict()` like we have in the past.

Let's predict `diabetes`, a dichotomous outcome, using all the other variables in our modified dataset.

First, fit a logistic regression model using all of the observations except the very first one. Then use your fitted model to predict whether your holdout case is positive or negative for diabetes. Remember that in logistic regression, the model output (before applying the sigmoid function) is in **log-odds**. If this output is positive, the predicted probability is greater than 50%; if it is negative, the predicted probability is less than 50%.

Compare your prediction to the actual response for the first observation that you initially excluded. Did your model correctly classify this observation?

In [4]:
idx <- 1
df_loov <- df %>% slice(-idx)
df_left <- df %>% slice(idx)

log_reg_1 <- glm(diabetes ~ pregnant + glucose + pressure + triceps + mass + pedigree + age, data = df_loov, family = "binomial")
summary(log_reg_1)

predict(log_reg_1, df_left, type="response")


Call:
glm(formula = diabetes ~ pregnant + glucose + pressure + triceps + 
    mass + pedigree + age, family = "binomial", data = df_loov)

Coefficients:
             Estimate Std. Error z value Pr(>|z|)    
(Intercept) -9.537281   0.993617  -9.599  < 2e-16 ***
pregnant     0.122949   0.043708   2.813 0.004909 ** 
glucose      0.035231   0.004240   8.308  < 2e-16 ***
pressure    -0.007469   0.010314  -0.724 0.468963    
triceps      0.006572   0.014744   0.446 0.655779    
mass         0.082720   0.023320   3.547 0.000389 ***
pedigree     1.304616   0.363719   3.587 0.000335 ***
age          0.025756   0.014017   1.838 0.066133 .  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 674.58  on 530  degrees of freedom
Residual deviance: 465.70  on 523  degrees of freedom
AIC: 481.7

Number of Fisher Scoring iterations: 5


1 
0.729487

So we just calculated a single iteration of leave-one-out cross validation. We used 531 rows of our data to fit a model to predict the outcome of the last row.

Below, use a `for` loop to iterate through the rest of your dataset doing the same thing. You will need to:
* Create a data frame `results` with two columns: one named `actual` which holds the true classification for each observation, and one named `predicted`, which should be filled with `NA`s. This is where you'll store the output of your loop.
* Create a loop that runs through each row of your data, pulls that observation out, trains your model on the remaining data, and then tests the fitted model on your test observation.
* Store your model *predictions* ("pos" or "neg" -- not the log-odds) in the `predicted` column of your `results` dataframe

After you run your loop, print the first few lines of `results`.

In [9]:
# Initialize `results` data frame
results <- data.frame(actual = df$diabetes,
                    predicted = rep(NA, 532))

#for loop
results <- results %>% mutate(predicted = lapply(1:nrow(df), function(i){
    idx <- i
    df_loov <- df %>% slice(-idx)
    df_left <- df %>% slice(idx)
    log_reg <- glm(diabetes ~ pregnant + glucose + pressure + triceps + mass + pedigree + age, data = df_loov, family = "binomial")
    output <- predict(log_reg, df_left, type = "response")
    output_final <- if_else(output > .50, 'pos', 'neg')
    return(output_final)
}))

head(results)

,actual,predicted
,<fct>,<list>
1,pos,pos
2,neg,neg
3,neg,neg
4,pos,pos
5,pos,neg
6,pos,pos


Now, calculate the overall error of your model. What proportion of cases were incorrectly classified?

In [11]:
mean(results$predicted != results$actual)
# 22.2% of models were incorrectly classified

[1] 0.2218045

----
## 3. Compare to `cv.glm` (3 points)

Now, let's compare this result to the `cv.glm` function. Using the tutorial as a guide, use `cv.glm` to run LOOCV on the data, using the same model (i.e. still using all of the variables to predict diabetes diagnosis).

Note that, because this is a `classification` problem and not a regression problem like in the tutorial, we need to adjust the `cost` argument of `cv.glm`. For more details, see the function documentation by running `?cv.glm`:

In [6]:
?cv.glm

cv.glm {boot},R Documentation
data,"A matrix or data frame containing the data. The rows should be cases and the columns correspond to variables, one of which is the response."
glmfit,"An object of class ""glm"" containing the results of a generalized linear model fitted to data."
cost,A function of two vector arguments specifying the cost function for the cross-validation. The first argument to cost should correspond to the observed responses and the second argument should correspond to the predicted or fitted responses from the generalized linear model. cost must return a non-negative scalar value. The default is the average squared error function.
K,The number of groups into which the data should be split to estimate the cross-validation prediction error. The value of K must be such that all groups are of approximately equal size. If the supplied value of K does not satisfy this criterion then it will be set to the closest integer which does and a warning is generated specifying the value of K used. The default is to set K equal to the number of observations in data which gives the usual leave-one-out cross-validation.
call,The original call to cv.glm.
K,The value of K used for the K-fold cross validation.
delta,A vector of length two. The first component is the raw cross-validation estimate of prediction error. The second component is the adjusted cross-validation estimate. The adjustment is designed to compensate for the bias introduced by not using leave-one-out cross-validation.
seed,The value of .Random.seed when cv.glm was called.


Here, we see `cost` is defined as:
> "A function of two vector arguments specifying the cost function for the cross-validation. The first argument to cost should correspond to the **observed responses** and the second argument should correspond to the **predicted or fitted responses** from the generalized linear model."

In the example code (scroll to bottom of the docs), we see that the appropriate cost function for a binary classification is

```
cost <- function(r, pi = 0) {
  mean(abs(r - pi) > 0.5)
}
```

Where `r` is the vector of observed responses (technically "pos" and "neg", but R treats these as 1 and 0 under the hood), and `pi` is the vector of *probabilities* (not log-odds) fit by the model. Thus, this boils down to our error: what proportion of observations were incorrectly classified. You will need to include this code below.

In [12]:
log_reg_2 <- glm(diabetes ~ pregnant + glucose + pressure + triceps + mass + pedigree + age, data = df, family = "binomial")

cost <- function(r, pi = 0) mean(abs(r-pi) > 0.5)

cv.glm(df, log_reg_2, cost)$delta[1]

[1] 0.2218045

How do your results compare to your manual LOOCV above?

> My results are the same as the manual LOOCV above. 


---
## 4. Adjusting K and Reflection (2 points)

Recall that LOOCV has some drawbacks. In particular, it has quite high *variance* which can lead to poor performance on new test data. We can reduce this variance by increasing K.

Below, re-run your cross validation using `cv.glm` with `k` set to 3, 5, 10, and 15.

In [8]:
set.seed(1)

# K = 3
K = 3
log_reg_2 <- glm(diabetes ~ pregnant + glucose + pressure + triceps + mass + pedigree + age, data = df, family = "binomial")
cost <- function(r, pi = 0) mean(abs(r-pi) > 0.5)
cv.glm(df, log_reg_2, cost, K)$delta[1]


# K = 5
K = 5
cv.glm(df, log_reg_2, cost, K)$delta[1]


# K = 10
K = 10
cv.glm(df, log_reg_2, cost, K)$delta[1]


# K = 15
K = 15
cv.glm(df, log_reg_2, cost, K)$delta[1]

[1] 0.2105263

[1] 0.2161654

[1] 0.2236842

[1] 0.2274436

#### Reflection

How do your errors compare to your LOOCV error above? How do they change as k increases?
> Errors increase as K increases. Overall, lower than the LOOCV error. 

If you change the random seed above, you'll get slightly different errors. If you were to do the same with your LOOCV above , would you expect to get different results each time? Why or why not?
> No, because the LOOCV pertains to the variability within the dataset, not each K being randomized and influenced set.seed. 

**DUE:** 11:59pm March 19, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> I did not collaborate with any of my classmates on this assignment. 

**GenAI Utilization** Did you utilize any generative AI tools on this assignment? If so, please list the item and the paste respective prompt you used.

> I did not utilize any generative AI tools on this assignment.